# RetailDemandML EDA

This notebook documents the exploratory checks behind the scripted EDA report. The script remains the source of truth for reproducible figures:

```bash
make eda
```

The notebook is intentionally lightweight, but it records the questions that affected feature engineering and validation design.

## Questions Checked

- Does the raw file match the expected Kaggle panel shape?
- How strong are day-of-week and month effects?
- Are store and item effects large enough to justify categorical and aggregate features?
- Which lag windows are useful without leaking the current target?
- What external demand drivers are missing from the public dataset?

In [ ]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path('..').resolve()
raw_path = ROOT / 'data' / 'raw' / 'train.csv'
summary_path = ROOT / 'reports' / 'eda_summary.json'

summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
summary

In [ ]:
df = pd.read_csv(raw_path, parse_dates=['date'])
df.shape, df['date'].min(), df['date'].max(), df['store'].nunique(), df['item'].nunique()

## Dataset Shape

The expected Kaggle training set is a complete daily panel: 10 stores, 50 items, and five years of daily sales. If the shape below is materially smaller, the local file is probably sample data and the public metrics should not be refreshed from it.

In [ ]:
panel = (
    df.groupby(['store', 'item'])['date']
    .agg(['min', 'max', 'count'])
    .reset_index()
)
panel['count'].describe()

## Seasonality

Day-of-week and month summaries are useful sanity checks before building lag features. The model should see these as calendar features instead of relying only on raw item/store identifiers.

In [ ]:
seasonality = df.assign(
    day_of_week=df['date'].dt.day_name(),
    month=df['date'].dt.month,
)
seasonality.groupby('day_of_week')['sales'].mean().sort_values(ascending=False)

In [ ]:
seasonality.groupby('month')['sales'].mean().round(2)

## Store and Item Effects

The project keeps store/item as categorical variables and also creates shifted historical aggregates. Those aggregates are fitted from prior observations only, which avoids using the target value from the row being scored.

In [ ]:
store_summary = df.groupby('store')['sales'].agg(['mean', 'median', 'std']).round(2)
item_summary = df.groupby('item')['sales'].agg(['mean', 'median', 'std']).round(2)
store_summary.head(), item_summary.sort_values('mean', ascending=False).head(10)

## Lag Signal

Lag correlations are computed within each store/item series and shifted in the feature pipeline. Weekly lags are expected to be strong because the raw data has visible weekly seasonality.

In [ ]:
ordered = df.sort_values(['store', 'item', 'date']).copy()
lag_corr = {}
for lag in [1, 7, 14, 28, 56]:
    ordered[f'sales_lag_{lag}'] = ordered.groupby(['store', 'item'])['sales'].shift(lag)
    lag_corr[lag] = ordered[['sales', f'sales_lag_{lag}']].corr().iloc[0, 1]
pd.Series(lag_corr, name='correlation').round(3)

## Validation and Missing Drivers

The holdout periods are chronological because demand is autocorrelated. The public dataset does not include price, promotions, stockouts, inventory, or local events, so unexplained residuals may reflect real business drivers that are absent from the data rather than model capacity alone.